# Config

In [1]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

: 

: 

In [2]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


: 

: 

: 

# 1) Split dataset 

In [ ]:
from utils.dataset import get_dataset_to_split, split_dataset
import pandas as pd
import numpy as np
import os

#Save data
path = "/tmp/final_project"
filepath = os.path.join(path, "datasets/features.csv")
df=pd.read_csv(filepath)

feat_col = "Categoria Disciplina"
df = get_dataset_to_split(df, feat_col)

#Eliminar elementos indefinidos 
df = df[df[feat_col].notna() & (df[feat_col] != "SIN AREA")]

#Eliminar duplicados
df = df.drop_duplicates("Código VRID")

#Split data and save idx
ids = np.array(df["Código VRID"])
labels = np.array(df[feat_col])
savepath = os.path.join(path, "dataSplits/areas_ocde/train_test_ids_3folds.json")
split_dataset(savepath, ids, labels)

Test size: 113
Fold 0 - Val size: 151
Archivo guardado exitosamente en /tmp/final_project/dataSplits/areas_ocde/train_test_ids_3folds.json


# 2) Prepare labels

In [3]:
import json
import pandas as pd

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/areas_ocde/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)

In [4]:
import pandas as pd
import os 
from preprocess.translate import translate_OCDE_features

path_trads = "/tmp/final_project/datasets"

#Asignar traducciones a areas OCDE
filePATH = os.path.join(path_trads, "ocde_area_traduction.csv")
df_translations = pd.read_csv(filePATH)
df["area_ocde"] = translate_OCDE_features(df, "Categoria Disciplina", df_translations)

#Asignar traducciones a subareas OCDE
filePATH = os.path.join(path_trads, "ocde_subarea_traduction.csv")
df_translations = pd.read_csv(filePATH)
df["sub_area_ocde"] = translate_OCDE_features(df, "Sub Area Disciplina", df_translations)

In [5]:
from preprocess.preprocess import one_hot_codification

comb_area = ["Natural Sciences",
                "Engineering and Technology",
                "Social Sciences",
                "Agricultural Sciences",
                "Humanities",
                "Medical and Health Sciences"]

# Crear labels 
df = one_hot_codification(df, "area_ocde", comb_area)

# 3) TF-IDF

In [16]:
from utils.dataset import gen_dataset_select_cols
from models.TIFD import gen_TFID_vectors
import numpy as np
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter
from utils.mlflow import eval_model
import warnings
from utils.save_results import save_models_and_metrics
from utils.save_results import model_to_pipeline

warnings.filterwarnings(
    "ignore",
    message="The objective has been evaluated at point",
    category=UserWarning,
    module="skopt.optimizer.optimizer"
)

savepath = os.path.join(path, "output/areas_ocde/TF_IDF")
split_idx_path = os.path.join(path, "dataSplits/areas_ocde/train_test_ids_3folds.json")

for des in comb_desafios:
    #Columnas a seleccionar para clasificación
    cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

    #Lectura de codigos VRID Test
    codes_test = dataset_index["Test"]
    X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                    test_col=str(des))

    #Lectura de codigos VRID Train
    codes_train = dataset_index["kfolds"]
    codes_train = np.array([i for fold in codes_train for i in fold])
    X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                        test_col=str(des))
    df_decode = df_train[["idx", "Código VRID"]]

    #Creacion de vectores TFID
    X_train, X_test, vectorizer = gen_TFID_vectors(X_train, X_test, return_vectorizer=True)
    print(X_train.shape, X_test.shape)

    # 1. Elegir modelos a probar
    model_keys = [
        'LogisticRegression',
        'RandomForestClassifier',
        'XGBClassifier',
        'SVC',
    ]

    # 2. Obtener el diccionario de modelos y parámetros
    est_params_dict = get_est_params_dict(model_keys)
    print("📊 train:", Counter(y_train))
    print("📊 test:", Counter(y_test))

    # 3. Ejecutar entrenamiento, validación y test con tus funciones
    n_iter=20
    sample_weight_On=True
    scoring='f1_macro'
    results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                    n_iter=n_iter, sample_weight_On = sample_weight_On)

    best_model = select_best_model(results_val, models_dicc)

    # 4. Mostrar resultados
    print("\n🔍 Validación:")
    for model, metrics in results_val.items():
        print(f"{model}: {metrics}")

    # Métricas por idioma
    lang_es = df_test["Español"]
    results_test, preds_test = {}, {}
    for name, model in models_dicc.items():
        print(name)
        results, preds = eval_model(model, X_test, y_test, lang_es)
        print(results)
        results_test[name] = results
        preds_test[name] = preds

    models_dicc_pipeline = model_to_pipeline(vectorizer, models_dicc)
    area_path = os.path.join(savepath, str(des).replace(" ", "_"))
    save_models_and_metrics(area_path, results_val, models_dicc_pipeline, df_test, y_test, results_test, preds_test, save_preds=True, mode_classification="binary")
    


(451, 11989) (113, 11989)
📊 train: Counter({0: 236, 1: 215})
📊 test: Counter({0: 59, 1: 54})
(451,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.72, 'std_test_score': 0.04}
RandomForestClassifier: {'mean_test_score': 0.69, 'std_test_score': 0.06}
XGBClassifier: {'mean_test_score': 0.65, 'std_test_score': 0.03}
SVC: {'mean_test_score': 0.73, 'std_test_score': 0.03}
LogisticRegression
{'accuracy': 0.6902654867256637, 'f1_macro': 0.6839287141372972, 'cm': array([[47, 12],
       [23, 31]]), 'precision': 0.7209302325581395, 'recall': 0.5740740740740741, 'f1_es': 0.8124999999999999, 'f1_en': 0.6519218486145014, 'cm_es': array([[21,  4],
       [ 2,  3]]), 'cm_en': array([[26,  8],
       [21, 28]])}
RandomForestClassifier
{'accuracy': 0.6194690265486725, 'f1_macro': 0.6164653879548505, 'cm': array([[40, 19],
       [24, 30]]), 'precision': 0.6122448979591837, 'recall': 0.55555

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:197: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:198: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds_test[model_name]


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC
(451, 11989) (113, 11989)
📊 train: Counter({0: 368, 1: 83})
📊 test: Counter({0: 92, 1: 21})
(451,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.68, 'std_test_score': 0.04}
RandomForestClassifier: {'mean_test_score': 0.57, 'std_test_score': 0.04}
XGBClassifier: {'mean_test_score': 0.58, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.64, 'std_test_score': 0.03}
LogisticRegression
{'accuracy': 0.8141592920353983, 'f1_macro': 0.6984369043080443, 'cm': array([[81, 11],
       [10, 11]]), 'precision': 0.5, 'recall': 0.5238095238095238, 'f1_es': 0.9681917211328976, 'f1_en': 0.7590361445783134, 'cm_es': array([[25,  1],
       [ 0,  4]]), 'cm_en': array([[56, 10],
       [10,  7]])}
RandomForestClassifier
{'accuracy': 0.7964601769911505, 'f1_macro': 0.5700578990901572, 'cm': array([[86,  6],
       [17,  4]]), 'pre

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:197: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:198: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds_test[model_name]


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC
(451, 11989) (113, 11989)
📊 train: Counter({0: 389, 1: 62})
📊 test: Counter({0: 97, 1: 16})
(451,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.78, 'std_test_score': 0.05}
RandomForestClassifier: {'mean_test_score': 0.74, 'std_test_score': 0.07}
XGBClassifier: {'mean_test_score': 0.72, 'std_test_score': 0.06}
SVC: {'mean_test_score': 0.77, 'std_test_score': 0.06}
LogisticRegression
{'accuracy': 0.9380530973451328, 'f1_macro': 0.8691480562448304, 'cm': array([[94,  3],
       [ 4, 12]]), 'precision': 0.8, 'recall': 0.75, 'f1_es': 0.8685185185185186, 'f1_en': 0.9564511414931965, 'cm_es': array([[16,  3],
       [ 1, 10]]), 'cm_en': array([[78,  0],
       [ 3,  2]])}
RandomForestClassifier
{'accuracy': 0.9203539823008849, 'f1_macro': 0.7976119402985076, 'cm': array([[96,  1],
       [ 8,  8]]), 'precision': 0.888

...

📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC
(451, 11989) (113, 11989)
📊 train: Counter({0: 421, 1: 30})
📊 test: Counter({0: 106, 1: 7})
(451,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.64, 'std_test_score': 0.06}
RandomForestClassifier: {'mean_test_score': 0.57, 'std_test_score': 0.07}
XGBClassifier: {'mean_test_score': 0.52, 'std_test_score': 0.07}
SVC: {'mean_test_score': 0.57, 'std_test_score': 0.07}
LogisticRegression
{'accuracy': 0.9380530973451328, 'f1_macro': 0.77737123557557, 'cm': array([[101,   5],
       [  2,   5]]), 'precision': 0.5, 'recall': 0.7142857142857143, 'f1_es': 0.8159090909090909, 'f1_en': 0.9819642205184376, 'cm_es': array([[19,  5],
       [ 1,  5]]), 'cm_en': array([[82,  0],
       [ 1,  0]])}
RandomForestClassifier
{'accuracy': 0.9292035398230089, 'f1_macro': 0.5814814814814815, 'cm': array([[104,   2],
       [  6,   1]])

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:197: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:198: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds_test[model_name]


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC
(451, 11989) (113, 11989)
📊 train: Counter({0: 428, 1: 23})
📊 test: Counter({0: 107, 1: 6})
(451,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.6, 'std_test_score': 0.09}
RandomForestClassifier: {'mean_test_score': 0.49, 'std_test_score': 0.0}
XGBClassifier: {'mean_test_score': 0.58, 'std_test_score': 0.08}
SVC: {'mean_test_score': 0.6, 'std_test_score': 0.09}
LogisticRegression
{'accuracy': 0.9380530973451328, 'f1_macro': 0.48401826484018257, 'cm': array([[106,   1],
       [  6,   0]]), 'precision': 0.0, 'recall': 0.0, 'f1_es': 0.9502824858757061, 'f1_en': 0.9045180722891567, 'cm_es': array([[29,  0],
       [ 1,  0]]), 'cm_en': array([[77,  1],
       [ 5,  0]])}
RandomForestClassifier
{'accuracy': 0.9469026548672567, 'f1_macro': 0.4863636363636364, 'cm': array([[107,   0],
       [  6,   0]]), 'precision': 

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:197: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:198: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds_test[model_name]


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC


### Inference

In [22]:
from utils.save_results import load_model

model_path = "/tmp/final_project/output/areas_ocde/TF_IDF/Natural_Sciences/models"
model_path = os.path.join(model_path)

inference_model = load_model(model_path)
sample = 'role of plant-bacterial interaction under over-irrigated conditions on the hydric relationships of plants and the maturation rates of kiwi fruits (delicous actinidia) from the regulation of the synthesis of ethylene precursors flooding, ethylene, plant water status, hypoxia, fruit quality, fruit softening over-watering is a very common practice in agriculture, despite the severe water crisis that affects chili. the effects generated by the lack of oxygenation in the roots of the plants are not usually considered by the farmers, since the high plasticity of the plant organisms to the different biotic and abiotic stresses can hide the real economic impact of the excessive application of water. the kiwi (actinidia delicious chev.) is a fruit of importance in chile, because we are the third exporter worldwide. unfortunately, because this fruit plant has its center of origin in the forests of monsoon climate of china, a large quantity of water is applied in the commercial orchards, many times up to the double the maximum water requirement of the cultivation. the kiwi is considered a fruit crop not tolerant to all-negmentation, and therefore, its mechanisms of escape or tolerance to the absence of oxygen of the roots due to the excess of water are poor in comparison with other fruit species. in this context, the synthesis and accumulation of ethylene in the plant organs has been associated with the physiological responses ki. . however, by allowing fruit to mature at 20°c for a week, the kiwis of over-regulated plants showed a higher level of softening and concentration of soluble solids than plants under optimal irrigation, clearly indicating a higher rate of maturation in the fruit of plants under abundant irrigation, and therefore a higher concentration of the ethylene in the fruits. against this, it is necessary to ask how it is possible that in a state of initial maturity there has not been differences in maturating between irrigation treatments, but once the fruit was harvested if they could be detected. a potential response falls in the synthesis of the precursors of ethylene, particularly the 1-aminopropane-1-carboxylic acid (acc), which has been found in higher concentrations in other tropical cultures susceptible to hypoxia and tropical origin. bacterial populations associated with the rizosphere and roots of plants, can be affected as a consequence of the anaerobic conditions generated by the neutralization. in particular the group of bacteria producing the enzyme acc-deaminase play an important role in the regulation of the effects of the soil and ethylene and roots.'
pred = inference_model.predict([sample])
print("Prediction:", pred)

Mejor modelo: LogisticRegression
Prediction: [0]


# 4) SPECTER

In [18]:
from utils.dataset import gen_dataset_select_cols
from models.specter import embed_texts
import numpy as np
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter
from utils.mlflow import eval_model
import warnings
from utils.save_results import save_models_and_metrics
from utils.save_results import model_to_pipeline
from models.specter import BERT_vectorizer

warnings.filterwarnings(
    "ignore",
    message="The objective has been evaluated at point",
    category=UserWarning,
    module="skopt.optimizer.optimizer"
)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

savepath = os.path.join(path, "output/areas_ocde/SPECTER")
split_idx_path = os.path.join(path, "dataSplits/areas_ocde/train_test_ids_3folds.json")

for area in comb_area:
    #Columnas a seleccionar para clasificación
    cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

    #Lectura de codigos VRID Test
    codes_test = dataset_index["Test"]
    X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                    test_col=str(area))

    #Lectura de codigos VRID Train
    codes_train = dataset_index["kfolds"]
    codes_train = np.array([i for fold in codes_train for i in fold])
    X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                        test_col=str(area))
    df_decode = df_train[["idx", "Código VRID"]]

    # Calcular embeddings
    # Parámetros para cargar modelo
    BASE_MODEL = "allenai/specter2_base"
    ADAPTER_NAME="allenai/specter2_classification"
    X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
    X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)
    print(X_train.shape, X_test.shape)

    # 1. Elegir modelos a probar
    model_keys = [
        'LogisticRegression',
        'RandomForestClassifier',
        'XGBClassifier',
        'SVC',
    ]

    # 2. Obtener el diccionario de modelos y parámetros
    est_params_dict = get_est_params_dict(model_keys)
    print("📊 train:", Counter(y_train))
    print("📊 test:", Counter(y_test))

    # 3. Ejecutar entrenamiento, validación y test con tus funciones
    n_iter=20
    sample_weight_On=True
    scoring='f1_macro'
    results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                    n_iter=n_iter, sample_weight_On = sample_weight_On)

    best_model = select_best_model(results_val, models_dicc)

    # 4. Mostrar resultados
    print("\n🔍 Validación:")
    for model, metrics in results_val.items():
        print(f"{model}: {metrics}")

    # Métricas por idioma
    lang_es = df_test["Español"]
    results_test, preds_test = {}, {}
    for name, model in models_dicc.items():
        print(name)
        results, preds = eval_model(model, X_test, y_test, lang_es)
        print(results)
        results_test[name] = results
        preds_test[name] = preds
    
    vectorizer = BERT_vectorizer(BASE_MODEL, ADAPTER_NAME)
    models_dicc_pipeline = model_to_pipeline(vectorizer, models_dicc)
    area_path = os.path.join(savepath, str(des).replace(" ", "_"))
    save_models_and_metrics(area_path, results_val, models_dicc_pipeline, df_test, y_test, results_test, preds_test, save_preds=True, mode_classification="binary")
    


/usr/local/lib/python3.10/dist-packages/torch/_utils.py:830: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
(451, 768) (113, 768)
📊 train: Counter({0: 428, 1: 23})
📊 test: Counter({0: 107, 1: 6})
(451,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.51, 'std_test_score': 0.03}
RandomForestClassifier: {'mean_test_score': 0.49, 'std_test_score': 0.0}
XGBClassifier: {'mean_test_score': 0.57, 'std_test_score': 0.07}
SVC: {'mean_test_score': 0.52, 'std_test_score': 0.07}
LogisticRegression
{'accuracy': 0.8938053097345132, 'f1_macro': 0.5431266846361186, 'cm': array([[100,   7],
       [  5,   1]]), 'precision': 0.125, 'recall': 0.16666666666666666, 'f1_es': 0.8976190476190476, 'f1_en': 0.9036144578313253, 'cm_es': array([[26,  3],
       [ 1,  0]]), 'cm_en': array([[74,  4],
       [ 4,  1]])}
RandomForestClassifier
{'accuracy': 0.7964601769911505, 'f1_macro': 0.4433497536945813, 'cm': array([[90, 17],
       [ 6,  0]]), 'precision': 0.0, 'recall

/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:197: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:198: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds_test[model_name]


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC


### Inference

In [23]:
from utils.save_results import load_model

model_path = "/tmp/final_project/output/areas_ocde/SPECTER/Natural_Sciences/models"
model_path = os.path.join(model_path)

inference_model = load_model(model_path)
sample = 'role of plant-bacterial interaction under over-irrigated conditions on the hydric relationships of plants and the maturation rates of kiwi fruits (delicous actinidia) from the regulation of the synthesis of ethylene precursors flooding, ethylene, plant water status, hypoxia, fruit quality, fruit softening over-watering is a very common practice in agriculture, despite the severe water crisis that affects chili. the effects generated by the lack of oxygenation in the roots of the plants are not usually considered by the farmers, since the high plasticity of the plant organisms to the different biotic and abiotic stresses can hide the real economic impact of the excessive application of water. the kiwi (actinidia delicious chev.) is a fruit of importance in chile, because we are the third exporter worldwide. unfortunately, because this fruit plant has its center of origin in the forests of monsoon climate of china, a large quantity of water is applied in the commercial orchards, many times up to the double the maximum water requirement of the cultivation. the kiwi is considered a fruit crop not tolerant to all-negmentation, and therefore, its mechanisms of escape or tolerance to the absence of oxygen of the roots due to the excess of water are poor in comparison with other fruit species. in this context, the synthesis and accumulation of ethylene in the plant organs has been associated with the physiological responses ki. . however, by allowing fruit to mature at 20°c for a week, the kiwis of over-regulated plants showed a higher level of softening and concentration of soluble solids than plants under optimal irrigation, clearly indicating a higher rate of maturation in the fruit of plants under abundant irrigation, and therefore a higher concentration of the ethylene in the fruits. against this, it is necessary to ask how it is possible that in a state of initial maturity there has not been differences in maturating between irrigation treatments, but once the fruit was harvested if they could be detected. a potential response falls in the synthesis of the precursors of ethylene, particularly the 1-aminopropane-1-carboxylic acid (acc), which has been found in higher concentrations in other tropical cultures susceptible to hypoxia and tropical origin. bacterial populations associated with the rizosphere and roots of plants, can be affected as a consequence of the anaerobic conditions generated by the neutralization. in particular the group of bacteria producing the enzyme acc-deaminase play an important role in the regulation of the effects of the soil and ethylene and roots.'
pred = inference_model.predict([sample])
print("Prediction:", pred)

Mejor modelo: LogisticRegression


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
Prediction: [0]
